# Day 3: Text Search

Now that we have documents (Day 1) and chunks (Day 2), we need a way to search them. We'll start with **text search** (also called lexical search), which uses exact keyword matching to find relevant chunks.

## What We're Building

A text search system that:
1. Indexes chunked documents using TF-IDF scoring
2. Supports field boosting (title matches rank higher than content matches)
3. Returns top-K results with relevance scores
4. Enables fast keyword-based retrieval

## Key Concepts

**TF-IDF (Term Frequency-Inverse Document Frequency)**: A statistical measure that evaluates how important a word is to a document in a collection. Words that appear frequently in a document but rarely across all documents get higher scores.

**Field Boosting**: Giving certain fields (like title) more weight than others (like content). A title match is often more relevant than a content match.

**Lexical vs Semantic Search**: Lexical search matches exact keywords ("machine learning" matches "machine learning"). Semantic search understands meaning ("ML" matches "machine learning"). We'll do semantic search on Day 4.

## Imports and Data Loading

We'll import functions from Day 1 (data ingestion) and Day 2 (chunking), then load and chunk our documents.

In [1]:
# Import from previous days
from day1 import read_repo_data
from day2 import chunk_sliding_window

# Import minsearch for text search
import minsearch

print("✓ All imports successful")

✓ read_repo_data() function defined and ready for testing
✓ All helper functions loaded
✓ Ready to process GitHub repositories

Next: Phase 4 will test with real repositories (DataTalks FAQ and Evidently docs)
TEST 1: DataTalks.Club FAQ Repository


Downloaded 26249310 bytes
Total files in archive: 1555
Markdown files found: 1285
Successfully processed 1285 documents

✓ Successfully processed 1285 documents

First 3 documents:

1. faq-main/.claude/commands/pr.md
   Metadata keys: ['description']
   Content length: 1875 characters
   Sample metadata: {'description': 'Review and process open FAQ PRs'}

2. faq-main/.claude/feedback.md
   Metadata keys: []
   Content length: 7027 characters

3. faq-main/.claude/refactoring-plan.md
   Metadata keys: []
   Content length: 7211 characters
TEST 2: Evidently AI Documentation


Downloaded 17545754 bytes
Total files in archive: 278
Markdown files found: 95
Successfully processed 95 documents

✓ Successfully processed 95 documents

First 3 documents:

1. docs-main/api-reference/endpoint/create.mdx
   Metadata keys: ['title', 'openapi']
   Content length: 0 characters
   Sample metadata: {'title': 'Create Plant', 'openapi': 'POST /plants'}

2. docs-main/api-reference/endpoint/delete.mdx
   Metadata keys: ['title', 'openapi']
   Content length: 0 characters
   Sample metadata: {'title': 'Delete Plant', 'openapi': 'DELETE /plants/{id}'}

3. docs-main/api-reference/endpoint/get.mdx
   Metadata keys: ['title', 'openapi']
   Content length: 0 characters
   Sample metadata: {'title': 'Get Plants', 'openapi': 'GET /plants'}


✓ All imports successful


In [2]:
# Load DataTalksClub FAQ
print("Loading DataTalksClub FAQ...")
datatalk_docs = read_repo_data("DataTalksClub", "faq")
print(f"Loaded {len(datatalk_docs)} documents")

Loading DataTalksClub FAQ...


Downloaded 26249310 bytes
Total files in archive: 1555
Markdown files found: 1285
Successfully processed 1285 documents
Loaded 1285 documents


In [3]:
# Load Evidently AI docs
print("Loading Evidently docs...")
evidently_docs = read_repo_data("evidentlyai", "docs")
print(f"Loaded {len(evidently_docs)} documents")

Loading Evidently docs...


Downloaded 17545754 bytes
Total files in archive: 278
Markdown files found: 95
Successfully processed 95 documents
Loaded 95 documents


In [4]:
# Chunk DataTalksClub FAQ using sliding window (2000 chars, 1000 overlap)
print("Chunking DataTalksClub FAQ...")
datatalk_chunks = []
for doc in datatalk_docs:
    chunks = chunk_sliding_window(doc, chunk_size=2000, overlap=1000)
    datatalk_chunks.extend(chunks)

print(f"Created {len(datatalk_chunks)} chunks from {len(datatalk_docs)} documents")

Chunking DataTalksClub FAQ...
Created 1503 chunks from 1285 documents


In [5]:
# Chunk Evidently docs using sliding window
print("Chunking Evidently docs...")
evidently_chunks = []
for doc in evidently_docs:
    chunks = chunk_sliding_window(doc, chunk_size=2000, overlap=1000)
    evidently_chunks.extend(chunks)

print(f"Created {len(evidently_chunks)} chunks from {len(evidently_docs)} documents")

Chunking Evidently docs...
Created 648 chunks from 95 documents


## Preparing Documents for Indexing

minsearch expects documents with consistent fields. We'll add a "title" field to each chunk (extracted from metadata or filename).

In [6]:
def prepare_for_indexing(chunks):
    """Add title field to chunks for consistent indexing.
    
    Args:
        chunks: List of chunk dicts with metadata
    
    Returns:
        List of chunks with 'title' field added
    """
    prepared = []
    for chunk in chunks:
        # Extract title from metadata or use filename
        title = chunk['metadata'].get('title', chunk['filename'])
        
        # Create new chunk dict with title field
        prepared_chunk = {
            'content': chunk['content'],
            'title': title,
            'filename': chunk['filename'],
            'chunk_id': chunk['chunk_id'],
            'metadata': chunk['metadata']
        }
        prepared.append(prepared_chunk)
    
    return prepared

# Prepare both datasets
datatalk_prepared = prepare_for_indexing(datatalk_chunks)
evidently_prepared = prepare_for_indexing(evidently_chunks)

print(f"Prepared {len(datatalk_prepared)} DataTalksClub chunks")
print(f"Prepared {len(evidently_prepared)} Evidently chunks")
print(f"\nSample prepared chunk:")
print(f"  title: {datatalk_prepared[0]['title']}")
print(f"  filename: {datatalk_prepared[0]['filename']}")
print(f"  chunk_id: {datatalk_prepared[0]['chunk_id']}")
print(f"  content preview: {datatalk_prepared[0]['content'][:100]}...")

Prepared 1503 DataTalksClub chunks
Prepared 648 Evidently chunks

Sample prepared chunk:
  title: faq-main/.claude/commands/pr.md
  filename: faq-main/.claude/commands/pr.md
  chunk_id: faq-main/.claude/commands/pr.md-chunk-0
  content preview: Go through all open pull requests one by one. For each PR:

## 1. Show Details
- PR number and title...


## Creating the Text Search Index

We'll use minsearch.Index with:
- **text_fields**: Fields to index for text search (content, title)
- **keyword_fields**: Fields for exact match filtering (filename, chunk_id)

Field boosting will be applied at search time (title:2.0, content:1.0).

In [7]:
# Create index for DataTalksClub FAQ
print("Creating DataTalksClub FAQ index...")
datatalk_index = minsearch.Index(
    text_fields=["content", "title"],
    keyword_fields=["filename", "chunk_id"]
)

# Fit index with prepared chunks
datatalk_index.fit(datatalk_prepared)
print(f"✓ Indexed {len(datatalk_prepared)} DataTalksClub chunks")

Creating DataTalksClub FAQ index...
✓ Indexed 1503 DataTalksClub chunks


In [8]:
# Create index for Evidently docs
print("Creating Evidently docs index...")
evidently_index = minsearch.Index(
    text_fields=["content", "title"],
    keyword_fields=["filename", "chunk_id"]
)

# Fit index with prepared chunks
evidently_index.fit(evidently_prepared)
print(f"✓ Indexed {len(evidently_prepared)} Evidently chunks")

Creating Evidently docs index...
✓ Indexed 648 Evidently chunks


## text_search() Function

We'll create a wrapper function that provides a consistent interface for text search across both datasets.

Field boosting is applied via `boost_dict={"title": 2.0, "content": 1.0}` - title matches get 2x weight because a title match is often more relevant than a content match.

In [9]:
def text_search(index, query: str, top_k: int = 5) -> list[dict]:
    """Search index using text/lexical matching with TF-IDF scoring and field boosting.
    
    Args:
        index: minsearch.Index instance
        query: Search query string
        top_k: Number of results to return (default 5)
    
    Returns:
        List of dicts with content, title, filename, chunk_id, metadata, score
        Sorted by relevance score (highest first)
    """
    results = index.search(
        query=query,
        boost_dict={"title": 2.0, "content": 1.0},
        num_results=top_k
    )
    return results

print("✓ text_search() function defined")

✓ text_search() function defined


## Test on DataTalksClub FAQ

Let's run queries where text search excels: exact terminology, specific course names, and technology names.

In [10]:
# Query 1: "machine learning" - exact terminology
print("=" * 80)
print("Query: 'machine learning'")
print("=" * 80)

results = text_search(datatalk_index, "machine learning", top_k=5)
print(f"\nFound {len(results)} results:\n")

for i, result in enumerate(results, 1):
    print(f"{i}. [Score: {result.get('score', 0):.3f}] {result['title']}")
    print(f"   File: {result['filename']}")
    print(f"   Preview: {result['content'][:150]}...")
    print()

Query: 'machine learning'

Found 5 results:

1. [Score: 0.000] faq-main/_questions/machine-learning-zoomcamp/misc/034_d767e50478_any-advice-for-adding-the-machine-learning-zoomcam.md
   File: faq-main/_questions/machine-learning-zoomcamp/misc/034_d767e50478_any-advice-for-adding-the-machine-learning-zoomcam.md
   Preview: I've seen LinkedIn users list DataTalksClub as Experience with titles such as:

- Machine Learning Fellow
- Machine Learning Student
- Machine Learnin...

2. [Score: 0.000] faq-main/_questions/machine-learning-zoomcamp/misc/023_840d7e7ee8_which-ide-is-recommended-for-machine-learning.md
   File: faq-main/_questions/machine-learning-zoomcamp/misc/023_840d7e7ee8_which-ide-is-recommended-for-machine-learning.md
   Preview: VSCode and Jupyter....

3. [Score: 0.000] faq-main/_questions/machine-learning-zoomcamp/module-9/001_07cc362868_where-is-the-model-for-week-9.md
   File: faq-main/_questions/machine-learning-zoomcamp/module-9/001_07cc362868_where-is-the-model-for-week-

In [11]:
# Query 2: "data engineering" - exact course name
print("=" * 80)
print("Query: 'data engineering'")
print("=" * 80)

results = text_search(datatalk_index, "data engineering", top_k=5)
print(f"\nFound {len(results)} results:\n")

for i, result in enumerate(results, 1):
    print(f"{i}. [Score: {result.get('score', 0):.3f}] {result['title']}")
    print(f"   File: {result['filename']}")
    print(f"   Preview: {result['content'][:150]}...")
    print()

Query: 'data engineering'

Found 5 results:

1. [Score: 0.000] faq-main/_questions/data-engineering-zoomcamp/general/038_86251abcdf_any-books-or-additional-resources-you-recommend.md
   File: faq-main/_questions/data-engineering-zoomcamp/general/038_86251abcdf_any-books-or-additional-resources-you-recommend.md
   Preview: Yes to both! Check out this document: [Awesome Data Engineering Resources](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/awesom...

2. [Score: 0.000] faq-main/_questions/data-engineering-zoomcamp/module-1/003_4ee5e16952_taxi-data-data-dictionary-for-ny-taxi-data.md
   File: faq-main/_questions/data-engineering-zoomcamp/module-1/003_4ee5e16952_taxi-data-data-dictionary-for-ny-taxi-data.md
   Preview: Yellow Trips: [Data Dictionary](https://www1.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf)

Green Trips: [Data Dictionary ...

3. [Score: 0.000] faq-main/_questions/data-engineering-zoomcamp/general/004_52217fc51b_course-i

In [12]:
# Query 3: "Docker" - specific technology name
print("=" * 80)
print("Query: 'Docker'")
print("=" * 80)

results = text_search(datatalk_index, "Docker", top_k=5)
print(f"\nFound {len(results)} results:\n")

for i, result in enumerate(results, 1):
    print(f"{i}. [Score: {result.get('score', 0):.3f}] {result['title']}")
    print(f"   File: {result['filename']}")
    print(f"   Preview: {result['content'][:150]}...")
    print()

Query: 'Docker'

Found 5 results:

1. [Score: 0.000] faq-main/_questions/mlops-zoomcamp/module-1/046_1bc7beaf5d_install-docker-in-wsl2-without-installing-docker-d.md
   File: faq-main/_questions/mlops-zoomcamp/module-1/046_1bc7beaf5d_install-docker-in-wsl2-without-installing-docker-d.md
   Preview: If you want to install Docker in WSL2 on Windows without Docker Desktop, follow these steps:

1. **Install Docker**

   You can ignore the warnings du...

2. [Score: 0.000] faq-main/_questions/mlops-zoomcamp/module-6/014_560c0eb368_managing-multiple-docker-containers-with-docker-co.md
   File: faq-main/_questions/mlops-zoomcamp/module-6/014_560c0eb368_managing-multiple-docker-containers-with-docker-co.md
   Preview: ### Problem Description

When a Docker Compose file contains many containers, running them all may consume too many resources. There is often a need t...

3. [Score: 0.000] faq-main/_questions/machine-learning-zoomcamp/module-5/008_371f4a6519_docker-cannot-connect-to-the-docker-d

### Why Text Search Works Here

Text search excels at:
- **Exact keyword matching**: "machine learning" matches documents containing exactly those words
- **Technical terminology**: "Docker", "data engineering" are precise terms with clear meanings
- **Proper names**: Course names, technology names, acronyms

The TF-IDF scoring ensures documents that use these terms frequently (but not too commonly across all documents) rank higher.

## Test on Evidently Docs

Let's test on technical documentation with domain-specific terminology.

In [13]:
# Query 1: "monitoring" - exact technical term
print("=" * 80)
print("Query: 'monitoring'")
print("=" * 80)

results = text_search(evidently_index, "monitoring", top_k=5)
print(f"\nFound {len(results)} results:\n")

for i, result in enumerate(results, 1):
    print(f"{i}. [Score: {result.get('score', 0):.3f}] {result['title']}")
    print(f"   File: {result['filename']}")
    print(f"   Preview: {result['content'][:150]}...")
    print()

Query: 'monitoring'

Found 5 results:

1. [Score: 0.000] Batch monitoring
   File: docs-main/docs/platform/monitoring_local_batch.mdx
   Preview: e section on [Alerts](/docs/platform/alerts).
  </Step>
</Steps>

<Tip>
  **Running Tests vs Reports**. Structuring your evaluations as Tests - as opp...

2. [Score: 0.000] Batch monitoring
   File: docs-main/docs/platform/monitoring_local_batch.mdx
   Preview:  </Step>
  <Step title="Run the evals">
    You must independently execute Reports on a chosen cadence. Consider tools like Airflow. You can send Repo...

3. [Score: 0.000] Batch monitoring
   File: docs-main/docs/platform/monitoring_local_batch.mdx
   Preview: Read the overview of the approach [here](/docs/platform/monitoring_overview).

![](/images/monitoring_flow_batch.png)

Batch monitoring relies on the ...

4. [Score: 0.000] Data drift
   File: docs-main/metrics/explainer_drift.mdx
   Preview: videntlyai.com/blog/ml-monitoring-do-i-need-data-drift)

* ["My data drifted. What's ne

In [14]:
# Query 2: "data drift" - specific concept
print("=" * 80)
print("Query: 'data drift'")
print("=" * 80)

results = text_search(evidently_index, "data drift", top_k=5)
print(f"\nFound {len(results)} results:\n")

for i, result in enumerate(results, 1):
    print(f"{i}. [Score: {result.get('score', 0):.3f}] {result['title']}")
    print(f"   File: {result['filename']}")
    print(f"   Preview: {result['content'][:150]}...")
    print()

Query: 'data drift'

Found 5 results:

1. [Score: 0.000] Data drift
   File: docs-main/metrics/explainer_drift.mdx
   Preview: ch as articles.

<Tip>
  **Text descriptors drift**. If you work with raw text data, you can also check for distribution drift in text descriptors (su...

2. [Score: 0.000] Data Drift
   File: docs-main/metrics/preset_data_drift.mdx
   Preview: e proxy metrics. If you detect drift in features or prediction, you can trigger labelling and retraining, or decide to pause and switch to a different...

3. [Score: 0.000] Data Drift
   File: docs-main/metrics/preset_data_drift.mdx
   Preview: cept Drift](https://www.evidentlyai.com/ml-in-production/concept-drift). To build intuition about different drift detection methods, check these resea...

4. [Score: 0.000] Data Drift
   File: docs-main/metrics/preset_data_drift.mdx
   Preview: Target value, it will be evaluated together with other columns.

* **Overall dataset drift.** Returns the share of drifting columns in the

In [15]:
# Query 3: "evidently" - product name
print("=" * 80)
print("Query: 'evidently'")
print("=" * 80)

results = text_search(evidently_index, "evidently", top_k=5)
print(f"\nFound {len(results)} results:\n")

for i, result in enumerate(results, 1):
    print(f"{i}. [Score: {result.get('score', 0):.3f}] {result['title']}")
    print(f"   File: {result['filename']}")
    print(f"   Preview: {result['content'][:150]}...")
    print()

Query: 'evidently'

Found 5 results:

1. [Score: 0.000] Evidently Cloud
   File: docs-main/docs/setup/cloud.mdx
   Preview: ## 1. Create an Account

- If not yet, sign up for a [free Evidently Cloud account](https://app.evidently.cloud/signup).
- After logging in, create an...

2. [Score: 0.000] Why Evidently?
   File: docs-main/faq/why_evidently.mdx
   Preview: We’re building Evidently AI to help teams ship reliable AI products: whether it’s an ML model, an LLM app, or a complex agent workflow.

Our tools are...

3. [Score: 0.000] Why Evidently?
   File: docs-main/faq/why_evidently.mdx
   Preview: modular and component-based, so you can start small: you don't have to deploy a service with multiple databases just to run a single eval.

- Start wi...

4. [Score: 0.000] Why Evidently?
   File: docs-main/faq/why_evidently.mdx
   Preview:  LLM use cases. From ranking metrics to data drift algorithms and LLM judges, we’ve done the hard work by implementing metrics and ways to visualize t...

## Text Search Limitations

Let's see where text search struggles: paraphrases and conceptual queries.

In [16]:
# Limitation 1: Paraphrase - "how to improve models" won't match "fine-tuning"
print("=" * 80)
print("Limitation Query: 'how to improve models'")
print("(Looking for content about fine-tuning, hyperparameter optimization)")
print("=" * 80)

results = text_search(datatalk_index, "how to improve models", top_k=5)
print(f"\nFound {len(results)} results:\n")

for i, result in enumerate(results, 1):
    print(f"{i}. [Score: {result.get('score', 0):.3f}] {result['title']}")
    print(f"   Preview: {result['content'][:150]}...")
    print()

print("❌ Text search limitation: No exact keyword overlap with 'fine-tuning' or 'optimization'")
print("   The query uses different words than the actual content.")

Limitation Query: 'how to improve models'
(Looking for content about fine-tuning, hyperparameter optimization)

Found 5 results:

1. [Score: 0.000] faq-main/_questions/machine-learning-zoomcamp/module-5/062_fca4671a69_pretrained-models-deployment.md
   Preview: Yes, you can use pre-trained models and focus on deployment, but you must include some training element (such as fine-tuning, retraining, or a compara...

2. [Score: 0.000] faq-main/_questions/machine-learning-zoomcamp/module-8/033_56e523f708_how-can-data-augmentation-improve-model-performanc.md
   Preview: Data augmentation artificially expands the training dataset by applying transformations like flipping, cropping, and adjusting brightness or contrast....

3. [Score: 0.000] faq-main/_questions/data-engineering-zoomcamp/module-4/079_78fda262c6_how-does-dbt-handle-dependencies-between-models.md
   Preview: Dbt provides a mechanism called `ref` to manage dependencies between models. By referencing other models using the `ref` ke

In [17]:
# Limitation 2: Conceptual query - "understanding data quality"
print("=" * 80)
print("Limitation Query: 'understanding data quality'")
print("(Looking for content about data validation, metrics, monitoring)")
print("=" * 80)

results = text_search(evidently_index, "understanding data quality", top_k=5)
print(f"\nFound {len(results)} results:\n")

for i, result in enumerate(results, 1):
    print(f"{i}. [Score: {result.get('score', 0):.3f}] {result['title']}")
    print(f"   Preview: {result['content'][:150]}...")
    print()

print("❌ Text search limitation: Query is conceptual, not using specific metric names")
print("   Documents might discuss quality without using the word 'understanding'.")

Limitation Query: 'understanding data quality'
(Looking for content about data validation, metrics, monitoring)

Found 5 results:

1. [Score: 0.000] Data stats and quality
   Preview: e target will be also shown in the matrix according to its type.

<img height="812" width="2050" src="https://docs.evidentlyai.com/~gitbook/image?url=...

2. [Score: 0.000] Data stats and quality
   Preview: 4508ESQxkbpOxj%252Fuploads%252Fgit-blob-9b4017bb0fd6a38fae0b868c2cf016a2ec8e2250%252Freports_data_quality_in_time_datetime.png%3Falt%3Dmedia&width=768...

3. [Score: 0.000] Data stats and quality
   Preview: gitbook/image?url=https%3A%2F%2F256125905-files.gitbook.io%2F%7E%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FeE67gM4508ESQxkbpOxj%2...

4. [Score: 0.000] Data stats and quality
   Preview: #### 1. Summary widget

The table gives an overview of the dataset, including missing or empty features and other general information. It also shows t...

5. [Score: 0.000] Data stats and qual

### Why Text Search Fails Here

Text search requires **exact keyword overlap**:
- "how to improve models" doesn't contain "fine-tuning" or "optimization"
- "understanding data quality" is too general - documents use specific terms like "drift detection" or "metric calculation"

**This is why we need semantic search** (Day 4): Vector embeddings can match "improve models" with "fine-tuning" because they understand the semantic relationship, even when keywords don't overlap.

## Day 3 Learnings Summary

### What is TF-IDF?

TF-IDF (Term Frequency-Inverse Document Frequency) scores words based on:
- **Term Frequency (TF)**: How often a word appears in a document
- **Inverse Document Frequency (IDF)**: How rare a word is across all documents

Words that appear frequently in a document but rarely across all documents get high scores. Common words like "the" or "is" get low scores.

### Why Field Boosting Matters

With `boost_dict={"title": 2.0, "content": 1.0}` at search time:
- A title match counts 2x more than a content match
- If a query term appears in the title, that document ranks higher
- This is especially useful for documentation where titles are highly descriptive

### When Text Search Works Best

✅ **Exact terminology**: "TF-IDF", "machine learning", "data engineering"  
✅ **Acronyms**: "API", "FAQ", "ML"  
✅ **Proper names**: "Docker", "Evidently", course names  
✅ **Technical terms**: "monitoring", "data drift", "embeddings"  

### When Text Search Fails

❌ **Paraphrases**: "how to improve models" ≠ "fine-tuning techniques"  
❌ **Conceptual queries**: "understanding data quality" ≠ specific metric names  
❌ **Synonyms**: "buy" ≠ "purchase", "start" ≠ "begin"  
❌ **Different vocabulary**: User query uses different words than document content  

### Preview: Day 4 - Semantic Search

Semantic search (vector search) solves the paraphrase problem:
- Converts text to embeddings (numerical vectors)
- Measures semantic similarity, not just keyword overlap
- "improve models" matches "fine-tuning" because embeddings capture meaning
- Best for conceptual queries, question answering, and when users don't know exact terminology

Day 5 will combine both approaches (hybrid search) for the best of both worlds.

## Vector Search (Semantic)

Now we'll add **semantic search** (also called vector search) to handle the paraphrases and conceptual queries that text search fails on.

### Key Concepts

**Embeddings**: Dense vector representations of text that capture semantic meaning. Similar concepts have similar vectors, even when keywords differ.

**all-MiniLM-L6-v2**: A lightweight embedding model (22MB, 384 dimensions) optimized for fast CPU inference. It generates embeddings at ~14.7ms per 1000 tokens.

**Cosine Similarity**: Measures the angle between two vectors. Value of 1.0 means identical direction (semantically identical), 0.0 means orthogonal (unrelated).

### Why Vector Search Matters

Text search requires exact keyword overlap:
- Query "how to improve models" won't match "fine-tuning techniques"
- Query "understanding data quality" won't match specific metric terms

Vector search captures meaning:
- "improve models" embeds near "fine-tuning", "optimization", "training"
- Conceptual queries find semantically related content

In [18]:
# Import for vector search
from sentence_transformers import SentenceTransformer
from minsearch import VectorSearch
import numpy as np
from pathlib import Path

# Create embeddings cache directory
embeddings_dir = Path("data/embeddings")
embeddings_dir.mkdir(parents=True, exist_ok=True)

print("✓ Vector search imports successful")
print(f"✓ Embeddings cache directory: {embeddings_dir}")

✓ Vector search imports successful
✓ Embeddings cache directory: data/embeddings


In [19]:
# Load all-MiniLM-L6-v2 model (downloads ~22MB on first run to ~/.cache/huggingface/)
print("Loading sentence-transformers model (all-MiniLM-L6-v2)...")
print("Note: First run downloads ~22MB model to ~/.cache/huggingface/")
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"✓ Model loaded: {model.get_sentence_embedding_dimension()}-dimensional embeddings")

Loading sentence-transformers model (all-MiniLM-L6-v2)...
Note: First run downloads ~22MB model to ~/.cache/huggingface/


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Model loaded: 384-dimensional embeddings


In [20]:
def generate_embeddings(chunks: list[dict], model, batch_size: int = 32) -> np.ndarray:
    """Generate embeddings for chunk content with progress tracking.

    Args:
        chunks: List of chunk dicts with 'content' key
        model: SentenceTransformer model instance
        batch_size: Chunks per batch (default 32 per D-01)

    Returns:
        numpy array of shape (N, 384) with embeddings
    """
    # Extract content from chunks
    texts = [chunk['content'] for chunk in chunks]

    # Generate embeddings with progress bar (per D-02)
    print(f"Generating embeddings for {len(texts)} chunks (batch_size={batch_size})...")
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True  # tqdm progress per D-02
    )

    return embeddings


def get_or_generate_embeddings(
    chunks: list[dict],
    cache_path: Path,
    model,
    batch_size: int = 32
) -> np.ndarray:
    """Load embeddings from cache or generate if not present.

    Args:
        chunks: List of chunk dicts with 'content' key
        cache_path: Path to .npy cache file
        model: SentenceTransformer model instance
        batch_size: Chunks per batch

    Returns:
        numpy array of embeddings
    """
    if cache_path.exists():
        print(f"Loading cached embeddings from {cache_path}...")
        embeddings = np.load(cache_path)
        print(f"✓ Loaded {embeddings.shape[0]} embeddings ({embeddings.shape[1]}-dim)")
        return embeddings

    # Generate new embeddings
    embeddings = generate_embeddings(chunks, model, batch_size)

    # Cache to disk (per D-05, D-06, D-07)
    np.save(cache_path, embeddings)
    print(f"✓ Cached embeddings to {cache_path}")
    print(f"  Shape: {embeddings.shape}")

    return embeddings

print("✓ Embedding functions defined")

✓ Embedding functions defined


In [21]:
# Generate or load cached embeddings for DataTalksClub FAQ (per D-06)
datatalk_cache = embeddings_dir / "datatalk_faq.npy"
datatalk_embeddings = get_or_generate_embeddings(
    datatalk_prepared,  # ~8,565 chunks from Phase 14
    datatalk_cache,
    model,
    batch_size=32
)
print(f"\nDataTalksClub FAQ: {datatalk_embeddings.shape[0]} embeddings ready")

Loading cached embeddings from data/embeddings/datatalk_faq.npy...
✓ Loaded 1503 embeddings (384-dim)

DataTalksClub FAQ: 1503 embeddings ready


In [22]:
# Generate or load cached embeddings for Evidently docs (per D-06)
evidently_cache = embeddings_dir / "evidently_docs.npy"
evidently_embeddings = get_or_generate_embeddings(
    evidently_prepared,  # ~648 chunks from Phase 14
    evidently_cache,
    model,
    batch_size=32
)
print(f"\nEvidently docs: {evidently_embeddings.shape[0]} embeddings ready")

Loading cached embeddings from data/embeddings/evidently_docs.npy...
✓ Loaded 648 embeddings (384-dim)

Evidently docs: 648 embeddings ready


### Embedding Cache Management

Embeddings are cached to `data/embeddings/` directory:
- `datatalk_faq.npy` - DataTalksClub FAQ embeddings
- `evidently_docs.npy` - Evidently docs embeddings

**Cache invalidation** (per D-08): Delete cache files manually when:
- Chunks change (re-run Day 2 chunking)
- Model changes (switch to different embedding model)

Cache hit: <1 second load time
Cache miss: ~3-4 minutes for 8K chunks (one-time computation)

### minsearch.VectorSearch Index

We'll use `minsearch.VectorSearch` to create vector indices for semantic search (per SEARCH-09).

The VectorSearch class stores:
- Pre-computed document embeddings
- Document metadata (for returning with results)

Then provides a `search()` method that computes cosine similarity against a query embedding.

In [23]:
# Create VectorSearch indices using minsearch (per SEARCH-09)
# VectorSearch uses fit(vectors, payload) API

# DataTalksClub FAQ vector index
datatalk_vector_index = VectorSearch(
    keyword_fields=["filename", "chunk_id"]
)
datatalk_vector_index.fit(datatalk_embeddings, datatalk_prepared)
print(f"✓ DataTalksClub VectorSearch index created: {len(datatalk_prepared)} documents")

# Evidently docs vector index
evidently_vector_index = VectorSearch(
    keyword_fields=["filename", "chunk_id"]
)
evidently_vector_index.fit(evidently_embeddings, evidently_prepared)
print(f"✓ Evidently VectorSearch index created: {len(evidently_prepared)} documents")

✓ DataTalksClub VectorSearch index created: 1503 documents
✓ Evidently VectorSearch index created: 648 documents


### vector_search() Wrapper Function

We'll create a wrapper function that matches the `text_search()` interface exactly (per D-11, ORG-01):
- Same return format: list of dicts with content, title, score, etc.
- Score is cosine similarity (0.0-1.0, per D-14)

The wrapper:
1. Encodes the query using the sentence-transformers model
2. Delegates to `VectorSearch.search()` for similarity computation
3. Returns results in the same format as text_search()

In [24]:
def vector_search(
    index: VectorSearch,
    query: str,
    model,
    top_k: int = 5
) -> list[dict]:
    """Search using minsearch.VectorSearch with cosine similarity.

    Matches text_search() interface exactly (per D-11, ORG-01, ORG-02).
    Uses minsearch.VectorSearch internally (per SEARCH-09).

    Args:
        index: VectorSearch index with pre-computed embeddings
        query: Search query string
        model: SentenceTransformer model for query encoding
        top_k: Number of results to return (default 5)

    Returns:
        List of dicts with content, title, filename, chunk_id, metadata, score
        Score is cosine similarity (0.0-1.0, higher is more similar)
    """
    from sklearn.metrics.pairwise import cosine_similarity
    
    # Encode query (per D-12: <50ms overhead acceptable)
    query_embedding = model.encode([query])[0]

    # Use VectorSearch.search() for similarity computation (per SEARCH-09)
    # Get results with IDs so we can compute scores
    results = index.search(query_embedding, num_results=top_k, output_ids=True)
    
    # Compute cosine similarity scores using index.vectors
    query_2d = query_embedding.reshape(1, -1)
    
    for result in results:
        result_id = result.pop('_id')  # Remove internal ID after using it
        # Get embedding from index.vectors (internal storage)
        result_embedding = index.vectors[result_id]
        # Compute cosine similarity
        score = cosine_similarity(query_2d, result_embedding.reshape(1, -1))[0][0]
        result['score'] = float(score)

    return results

print("✓ vector_search() wrapper function defined")
print("  Signature: vector_search(index, query, model, top_k=5)")
print("  Uses: minsearch.VectorSearch internally")
print("  Returns: list[dict] with score as cosine similarity (0.0-1.0)")

✓ vector_search() wrapper function defined
  Signature: vector_search(index, query, model, top_k=5)
  Uses: minsearch.VectorSearch internally
  Returns: list[dict] with score as cosine similarity (0.0-1.0)


## Vector Search: Where Text Search Failed

Remember our text search limitations from earlier:
- "how to improve models" couldn't find "fine-tuning" content
- "understanding data quality" couldn't match specific metric terms

Let's see if vector search handles these paraphrases better (per D-16, D-17).

In [25]:
# Success Query 1: "how to improve models" - text search FAILED on this (per D-17)
print("=" * 80)
print("Query: 'how to improve models'")
print("(Text search failed - no exact keyword match with 'fine-tuning')")
print("=" * 80)

results = vector_search(
    datatalk_vector_index,
    "how to improve models",
    model,
    top_k=5
)

print(f"\nVector Search found {len(results)} results:\n")
for i, result in enumerate(results, 1):
    print(f"{i}. [Score: {result['score']:.3f}] {result['title']}")
    print(f"   File: {result['filename']}")
    print(f"   Preview: {result['content'][:150]}...")
    print()

print("✓ Vector search SUCCESS: Semantic similarity finds related content")
print("  even without exact keyword overlap ('improve' → 'fine-tuning', 'training')")

Query: 'how to improve models'
(Text search failed - no exact keyword match with 'fine-tuning')

Vector Search found 5 results:

1. [Score: 0.392] faq-main/_questions/machine-learning-zoomcamp/projects/008_9a458a7035_how-many-models-should-i-train.md
   File: faq-main/_questions/machine-learning-zoomcamp/projects/008_9a458a7035_how-many-models-should-i-train.md
   Preview: Regarding Point 4 in the midterm deliverables, which states, "Train multiple models, tune their performance, and select the best model," you might won...

2. [Score: 0.378] faq-main/_questions/machine-learning-zoomcamp/module-8/033_56e523f708_how-can-data-augmentation-improve-model-performanc.md
   File: faq-main/_questions/machine-learning-zoomcamp/module-8/033_56e523f708_how-can-data-augmentation-improve-model-performanc.md
   Preview: Data augmentation artificially expands the training dataset by applying transformations like flipping, cropping, and adjusting brightness or contrast....

3. [Score: 0.351] faq-main/

In [26]:
# Success Query 2: "understanding data quality" - text search FAILED on this
print("=" * 80)
print("Query: 'understanding data quality'")
print("(Text search failed - too conceptual, didn't match metric names)")
print("=" * 80)

results = vector_search(
    evidently_vector_index,
    "understanding data quality",
    model,
    top_k=5
)

print(f"\nVector Search found {len(results)} results:\n")
for i, result in enumerate(results, 1):
    print(f"{i}. [Score: {result['score']:.3f}] {result['title']}")
    print(f"   File: {result['filename']}")
    print(f"   Preview: {result['content'][:150]}...")
    print()

print("✓ Vector search SUCCESS: Conceptual understanding captures meaning")
print("  'data quality' semantically matches monitoring, validation, metrics content")

Query: 'understanding data quality'
(Text search failed - too conceptual, didn't match metric names)

Vector Search found 5 results:

1. [Score: 0.404] Introduction
   File: docs-main/docs/library/overview.mdx
   Preview: s:

| **Type**                  | **Example checks**                                                        |
| ------------------------- | ----------...

2. [Score: 0.385] All Metrics
   File: docs-main/metrics/all_metrics.mdx
   Preview: t equal to reference.</li></ul>          |

### Dataset data quality

Dataset-level data quality metrics.

<Info>
  [Data definition](/docs/library/da...

3. [Score: 0.376] Introduction
   File: docs-main/docs/library/overview.mdx
   Preview: t assesses a specific quality of a given text. It’s different from metrics (like accuracy or precision) that give a score for an entire _dataset_. You...

4. [Score: 0.356] All Metrics
   File: docs-main/metrics/all_metrics.mdx
   Preview:                                        |
| **ValueDrift

In [27]:
# Additional Success: Semantic synonyms
print("=" * 80)
print("Query: 'ML training tips'")
print("(Testing semantic similarity: 'ML' ≈ 'machine learning', 'tips' ≈ 'advice')")
print("=" * 80)

results = vector_search(
    datatalk_vector_index,
    "ML training tips",
    model,
    top_k=5
)

print(f"\nVector Search found {len(results)} results:\n")
for i, result in enumerate(results, 1):
    print(f"{i}. [Score: {result['score']:.3f}] {result['title']}")
    print(f"   File: {result['filename']}")
    print(f"   Preview: {result['content'][:150]}...")
    print()

Query: 'ML training tips'
(Testing semantic similarity: 'ML' ≈ 'machine learning', 'tips' ≈ 'advice')

Vector Search found 5 results:

1. [Score: 0.489] faq-main/_questions/machine-learning-zoomcamp/module-4/001_afd915ae5b_module-4-evaluation-overview.md
   File: faq-main/_questions/machine-learning-zoomcamp/module-4/001_afd915ae5b_module-4-evaluation-overview.md
   Preview: In Module 4, you’ll learn the core evaluation concepts used in classification tasks. Topics include:
- Metrics and diagnostics: precision, recall, ROC...

2. [Score: 0.460] faq-main/_questions/machine-learning-zoomcamp/projects/007_9f4011849d_acceptable-datasets-projects.md
   File: faq-main/_questions/machine-learning-zoomcamp/projects/007_9f4011849d_acceptable-datasets-projects.md
   Preview: Choose something non-toy (e.g., at least 100 rows) that lets you demonstrate the full ML pipeline: data preparation → modeling → evaluation → deployme...

3. [Score: 0.444] faq-main/_questions/machine-learning-zoomcamp/gener

## Vector Search Limitations

Vector search isn't perfect. Let's demonstrate where it struggles (per D-18, D-19):

1. **Acronym collapse**: Similar acronyms embed similarly even when they're different concepts
2. **Exact codes/IDs**: Specific identifiers like CVE numbers may not distinguish semantically
3. **Precise terminology**: When exact keywords matter, text search may be better

In [28]:
# Limitation 1: Exact terminology - text search may be better
print("=" * 80)
print("Limitation Query: 'TF-IDF'")
print("(Exact technical acronym - text search excels here)")
print("=" * 80)

# Compare vector search
print("\nVector Search Results:")
results = vector_search(
    datatalk_vector_index,
    "TF-IDF",
    model,
    top_k=3
)
for i, result in enumerate(results, 1):
    print(f"{i}. [Score: {result['score']:.3f}] {result['title']}")
    print(f"   Preview: {result['content'][:100]}...")

# Compare with text search
print("\n\nText Search Results:")
text_results = text_search(datatalk_index, "TF-IDF", top_k=3)
for i, result in enumerate(text_results, 1):
    print(f"{i}. [Score: {result.get('score', 0):.3f}] {result['title']}")
    print(f"   Preview: {result['content'][:100]}...")

print("\n❌ Vector search limitation: Exact acronyms may rank differently")
print("   Text search guarantees exact keyword match appears in results")

Limitation Query: 'TF-IDF'
(Exact technical acronym - text search excels here)

Vector Search Results:
1. [Score: 0.324] faq-main/_questions/llm-zoomcamp/module-2/004_db78580409_what-is-the-cosine-similarity.md
   Preview: Cosine similarity is a measure used to calculate the similarity between two non-zero vectors, often ...
2. [Score: 0.295] faq-main/_questions/llm-zoomcamp/general/013_de32098d49_what-other-alternatives-to-elasticsearch-are-there.md
   Preview: You could use some of these free alternatives to Elasticsearch:

- [Milvus](https://milvus.io/): An ...
3. [Score: 0.268] faq-main/_questions/data-engineering-zoomcamp/module-6/026_6e251d34b6_how-can-i-read-a-small-number-of-rows-from-the-par.md
   Preview: To read a small number of rows from a parquet file, you can use PyArrow or Apache Spark:

### Using ...


Text Search Results:
1. [Score: 0.000] faq-main/_questions/machine-learning-zoomcamp/module-9/032_8d914fcbe3_python-312-vs-tf-lite-217.md
   Preview: The latest versions

In [29]:
# Limitation 2: Similar acronyms collapse semantically
from sklearn.metrics.pairwise import cosine_similarity

print("=" * 80)
print("Limitation Query: Comparing 'API' vs 'FAQ' embeddings")
print("(Different concepts but similar acronym structure)")
print("=" * 80)

# Show that acronyms can have similar embeddings
api_embedding = model.encode(["API"])[0]
faq_embedding = model.encode(["FAQ"])[0]
similarity = cosine_similarity(
    api_embedding.reshape(1, -1),
    faq_embedding.reshape(1, -1)
)[0][0]

print(f"\nCosine similarity between 'API' and 'FAQ': {similarity:.3f}")
print("(Relatively high similarity despite being unrelated concepts)")

# Compare to truly related terms
ml_embedding = model.encode(["machine learning"])[0]
ai_embedding = model.encode(["artificial intelligence"])[0]
ml_ai_sim = cosine_similarity(
    ml_embedding.reshape(1, -1),
    ai_embedding.reshape(1, -1)
)[0][0]

print(f"Cosine similarity between 'machine learning' and 'artificial intelligence': {ml_ai_sim:.3f}")
print("(Higher similarity for truly related concepts)")

print("\n❌ Vector search limitation: Short acronyms may have unexpectedly")
print("   similar embeddings even when concepts are unrelated")

Limitation Query: Comparing 'API' vs 'FAQ' embeddings
(Different concepts but similar acronym structure)



Cosine similarity between 'API' and 'FAQ': 0.202
(Relatively high similarity despite being unrelated concepts)
Cosine similarity between 'machine learning' and 'artificial intelligence': 0.703
(Higher similarity for truly related concepts)

❌ Vector search limitation: Short acronyms may have unexpectedly
   similar embeddings even when concepts are unrelated


## Vector Search Summary

### When Vector Search Excels ✓

- **Paraphrases**: "improve models" finds "fine-tuning" content
- **Conceptual queries**: "understanding data quality" matches monitoring docs
- **Synonyms**: "ML" matches "machine learning" content
- **Natural language questions**: How/what/why queries

### When Text Search is Better ✗

- **Exact acronyms**: "TF-IDF" - need exact match, not semantically similar terms
- **Specific codes**: CVE numbers, version identifiers, product codes
- **Precise terminology**: When you know the exact term in the document

### Preview: Hybrid Search (Day 4)

The best approach combines both:
- Use text search for exact keyword matching
- Use vector search for semantic understanding
- Reciprocal Rank Fusion (RRF) merges results intelligently

This is called **hybrid search** and gives the best of both worlds.

## Hybrid Search (RRF Fusion)

Now we'll combine text and vector search using **Reciprocal Rank Fusion (RRF)** to get the best of both worlds.

### Key Concepts

**Reciprocal Rank Fusion (RRF)**: A parameter-free algorithm for combining ranked results from multiple retrieval methods. It operates on rank positions (1, 2, 3...) rather than raw scores, elegantly solving the score normalization problem.

**RRF Formula**: `score = 1 / (k + rank)` where k=60 is the industry-standard constant from Cormack et al. (SIGIR 2009). Documents appearing in multiple result lists get scores from both:

```
Document at rank 3 in text + rank 5 in vector:
RRF_score = 1/(60+3) + 1/(60+5) = 0.0159 + 0.0154 = 0.0313
```

**Deduplication**: Same document appearing in both text and vector results gets RRF scores from both, then deduplicated by chunk_id.

**Why Not Weighted Linear Combination?** TF-IDF scores (0-100+) and cosine similarity (0-1) have incompatible ranges. RRF uses only rank positions, eliminating normalization challenges.

In [ ]:
def hybrid_search(
    text_index,
    vector_index,
    query: str,
    model,
    top_k: int = 5
) -> list[dict]:
    """Hybrid search combining text and vector results with RRF fusion.

    Uses Reciprocal Rank Fusion (RRF) with k=60 to combine ranked results from
    text search (TF-IDF) and vector search (cosine similarity) into a unified
    ranking. Deduplicates by chunk_id and normalizes scores to 0.0-1.0 range.

    Args:
        text_index: minsearch.Index for text search
        vector_index: minsearch.VectorSearch for vector search
        query: Search query string
        model: SentenceTransformer for query encoding
        top_k: Number of results to return (default 5)

    Returns:
        List of dicts with content, title, filename, chunk_id, metadata, score
        Sorted by RRF score descending (0.0-1.0 range)
    """
    # 1. Get results from both methods (per D-10)
    text_results = text_search(text_index, query, top_k=top_k)
    vector_results = vector_search(vector_index, query, model, top_k=top_k)

    # 2. Compute RRF scores (per D-02: score = 1/(k + rank))
    k = 60  # Hardcoded per D-01, D-03 (industry standard from SIGIR 2009)
    rrf_scores = {}

    for rank, result in enumerate(text_results, start=1):
        chunk_id = result['chunk_id']
        rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0.0) + 1.0 / (k + rank)

    for rank, result in enumerate(vector_results, start=1):
        chunk_id = result['chunk_id']
        rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0.0) + 1.0 / (k + rank)

    # 3. Build unified results and deduplicate by chunk_id (per D-05, D-06, D-08)
    all_results = {}
    for result in text_results + vector_results:
        chunk_id = result['chunk_id']
        if chunk_id not in all_results:
            all_results[chunk_id] = result.copy()

    # 4. Attach RRF scores and normalize to 0.0-1.0 range (per D-12)
    for chunk_id, result in all_results.items():
        result['score'] = rrf_scores[chunk_id]

    # Normalize scores to 0.0-1.0 range
    if rrf_scores:
        max_score = max(rrf_scores.values())
        for result in all_results.values():
            result['score'] = result['score'] / max_score

    # 5. Sort by RRF score descending (per D-13)
    sorted_results = sorted(
        all_results.values(),
        key=lambda x: x['score'],
        reverse=True
    )

    # 6. Return top_k results (per D-09)
    return sorted_results[:top_k]

print("✓ hybrid_search() function defined")
print("  Uses RRF with k=60 for rank-based score fusion")
print("  Deduplicates by chunk_id, normalizes scores to 0.0-1.0")

### Test Hybrid Search

Let's test hybrid search on both datasets to verify it works correctly.

In [ ]:
# Test hybrid search on DataTalksClub FAQ
print("=" * 80)
print("Hybrid Search Test: DataTalksClub FAQ")
print("Query: 'machine learning'")
print("=" * 80)

results = hybrid_search(
    datatalk_index,
    datatalk_vector_index,
    "machine learning",
    model,
    top_k=5
)

print(f"\nFound {len(results)} results:\n")
for i, result in enumerate(results, 1):
    print(f"{i}. [RRF Score: {result['score']:.3f}] {result['title']}")
    print(f"   File: {result['filename']}")
    print(f"   Preview: {result['content'][:120]}...")
    print()

print("✓ Hybrid search combines text (exact keywords) and vector (semantic)")

In [ ]:
# Test hybrid search on Evidently docs
print("=" * 80)
print("Hybrid Search Test: Evidently Docs")
print("Query: 'data drift'")
print("=" * 80)

results = hybrid_search(
    evidently_index,
    evidently_vector_index,
    "data drift",
    model,
    top_k=5
)

print(f"\nFound {len(results)} results:\n")
for i, result in enumerate(results, 1):
    print(f"{i}. [RRF Score: {result['score']:.3f}] {result['title']}")
    print(f"   File: {result['filename']}")
    print(f"   Preview: {result['content'][:120]}...")
    print()